# Music Annotation Project V7.1: Fully Resumable Real Reconstruction Benchmark

CPU notebook. Re-run from the top after an interruption.

This benchmark uses the genuine aligned annotated page as reconstruction input. Clean pages are used only for evaluation. Every page/dilation reconstruction and evaluation record is checkpointed atomically to Google Drive. Biharmonic and Telea are each calculated once and reused by the hybrid methods.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


## Configuration, paths and durable state


In [ ]:
import cv2, json, hashlib, shutil, math, time, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from skimage.restoration import inpaint as sk_inpaint
from skimage.metrics import structural_similarity as ssim

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'; OBSERVED_DIR=ROOT/'aligned'; MASK_DIR=ROOT/'masks'/'corrected'
OUTPUT_ROOT=ROOT/'reconstruction_benchmark_v7_1_real'
CACHE=OUTPUT_ROOT/'persistent_cache'; RECON_CACHE=CACHE/'reconstruction'; METRIC_CACHE=CACHE/'evaluation'
STATE_PATH=OUTPUT_ROOT/'benchmark_state.json'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S'); OUT=OUTPUT_ROOT/'runs'/RUN_NAME
TABLES=OUT/'tables'; CONTACTS=OUT/'contact_sheets'; FAILURES=OUT/'failure_crops'
LOCAL=Path('/content/reconstruction_benchmark_v7_1'); LCLEAN=LOCAL/'clean'; LOBSERVED=LOCAL/'observed'; LMASK=LOCAL/'masks'
for d in [OUTPUT_ROOT,CACHE,RECON_CACHE,METRIC_CACHE,OUT,TABLES,CONTACTS,FAILURES,LCLEAN,LOBSERVED,LMASK]: d.mkdir(parents=True,exist_ok=True)

DILATIONS=[1,2]; CROP_MARGIN=48; MIN_COMPONENT_AREA=8; MIN_METRIC_COMPONENT_AREA=20
PAPER_SIGMA=45.0; PAPER_DARK_REJECTION=200; STRUCTURE_DARK_THRESHOLD=190
STRUCTURE_EDGE_LOW=70; STRUCTURE_EDGE_HIGH=170; STRUCTURE_GUARD_RADIUS=5
BOOTSTRAP_SAMPLES=2000; MAX_CONTACT_PAGES=24; MAX_FAILURE_CROPS=20
METHODS=['biharmonic','telea','paper_fill','paper_biharmonic_hybrid','paper_telea_hybrid']
BASELINE='biharmonic'
FORCE_REBUILD_RECONSTRUCTION=False
FORCE_REBUILD_EVALUATION=False


## Atomic helpers, discovery and local staging


In [ ]:
def image_files(folder):
    folder=Path(folder); return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []
def maps(folder): return {p.stem:p for p in image_files(folder)}
def find_local(folder,page):
    hits=[p for p in image_files(folder) if p.stem.lower()==page.lower()]; return hits[0] if len(hits)==1 else None
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(v): return hashlib.md5(json.dumps(v,sort_keys=True).encode()).hexdigest()
def read_json(path,default):
    try: return json.loads(Path(path).read_text()) if Path(path).exists() else default
    except Exception: return default
def write_json_atomic(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,allow_nan=True,default=str)); tmp.replace(path)
def save_npz_atomic(path,**arrays):
    path=Path(path); tmp=Path(str(path)+'.tmp.npz'); np.savez_compressed(tmp,**arrays); tmp.replace(path)
def valid_npz(path,required):
    try:
        with np.load(path) as z: return set(required).issubset(z.files)
    except Exception: return False
def copy_current(src,dst):
    if not dst.exists() or md5(src)!=md5(dst): shutil.copy2(src,dst)

state=read_json(STATE_PATH,{'version':'7.1','items':{},'failures':[]})
state['version']='7.1'; state.setdefault('items',{}); state.setdefault('failures',[])
a,b,c=maps(CLEAN_DIR),maps(OBSERVED_DIR),maps(MASK_DIR)
discovered=sorted(set(a)&set(b)&set(c)); valid_pages=[]; issues=[]
for page in discovered:
    clean=cv2.imread(str(a[page]),0); observed=cv2.imread(str(b[page]),0); mask=cv2.imread(str(c[page]),0)
    if clean is None or observed is None or mask is None: issues.append({'page':page,'reason':'unreadable'}); continue
    if clean.shape!=observed.shape or clean.shape!=mask.shape: issues.append({'page':page,'reason':'shape mismatch'}); continue
    if not np.any(mask>0): issues.append({'page':page,'reason':'empty mask'}); continue
    for src,dst in [(a[page],LCLEAN),(b[page],LOBSERVED),(c[page],LMASK)]: copy_current(src,dst/(page+src.suffix.lower()))
    valid_pages.append(page)
pd.DataFrame(issues).to_csv(TABLES/'discovery_issues.csv',index=False)
if not valid_pages: raise RuntimeError('No valid clean/observed/mask triplets.')
print('Valid pages:',len(valid_pages),'Issues:',len(issues),'State:',STATE_PATH)


## Leakage-safe, duplicate-free reconstruction functions


In [ ]:
def binary(mask): return np.where(mask>0,255,0).astype(np.uint8)
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]; assert np.array_equal(out[mask==0],original[mask==0]); return out
def prepare_mask(seed,dilation):
    mask=binary(seed)
    if dilation>0: mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return mask
def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8); h,w=mask.shape; boxes=[]
    for i in range(1,n):
        x,y,bw,bh,area=stats[i]
        if area>=MIN_COMPONENT_AREA: boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0); ay0=min(ay0,by0); ax1=max(ax1,bx1); ay1=max(ay1,by1); changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes
def crop_inpaint(observed,mask,method):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=observed[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
        if method=='telea': candidate=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA)
        elif method=='biharmonic':
            z=sk_inpaint.inpaint_biharmonic(im.astype(np.float32)/255.0,mk>0,channel_axis=None)
            candidate=np.clip(z*255,0,255).astype(np.uint8)
        local=result[y0:y1,x0:x1]; local[mk>0]=candidate[mk>0]
    return immutable(observed,result,mask)
def robust_paper_surface(observed,mask):
    visible=(mask==0)&(observed>=PAPER_DARK_REJECTION); values=observed.astype(np.float32)*visible; weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),PAPER_SIGMA); den=cv2.GaussianBlur(weights,(0,0),PAPER_SIGMA)
    surface=num/np.maximum(den,1e-4); fallback=cv2.GaussianBlur(observed,(0,0),PAPER_SIGMA)
    return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def visible_structure_guard(observed,mask):
    visible=observed.copy(); visible[mask>0]=255; dark=visible<STRUCTURE_DARK_THRESHOLD
    edges=cv2.Canny(visible,STRUCTURE_EDGE_LOW,STRUCTURE_EDGE_HIGH)>0
    horizontal=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*STRUCTURE_GUARD_RADIUS+1,2*STRUCTURE_GUARD_RADIUS+1))
    return cv2.dilate((dark|edges|horizontal).astype(np.uint8),k)>0
def reconstruct_production(observed,mask):
    biharmonic=crop_inpaint(observed,mask,'biharmonic')
    telea=crop_inpaint(observed,mask,'telea')
    paper=immutable(observed,robust_paper_surface(observed,mask),mask)
    guard=visible_structure_guard(observed,mask)&(mask>0)
    bi_hybrid=paper.copy(); bi_hybrid[guard]=biharmonic[guard]
    te_hybrid=paper.copy(); te_hybrid[guard]=telea[guard]
    return {'biharmonic':biharmonic,'telea':telea,'paper_fill':paper,
            'paper_biharmonic_hybrid':immutable(observed,bi_hybrid,mask),
            'paper_telea_hybrid':immutable(observed,te_hybrid,mask)}
for fn in [robust_paper_surface,visible_structure_guard,reconstruct_production]:
    if {'clean','target','ground_truth','reference'} & set(inspect.signature(fn).parameters): raise RuntimeError('Leakage-prone signature')
print('Leakage and immutability kernel ready.')


## Phase A: atomic reconstruction checkpoints


In [ ]:
settings={'version':'7.1','methods':METHODS,'dilations':DILATIONS,'crop_margin':CROP_MARGIN,'min_component_area':MIN_COMPONENT_AREA,
          'paper_sigma':PAPER_SIGMA,'paper_dark_rejection':PAPER_DARK_REJECTION,'structure_dark_threshold':STRUCTURE_DARK_THRESHOLD,
          'structure_edges':[STRUCTURE_EDGE_LOW,STRUCTURE_EDGE_HIGH],'structure_guard_radius':STRUCTURE_GUARD_RADIUS}
settings_hash=object_hash(settings); production_rows=[]
for pi,page in enumerate(valid_pages,1):
    op=find_local(LOBSERVED,page); mp=find_local(LMASK,page); cp=find_local(LCLEAN,page)
    observed=cv2.imread(str(op),0); seed=cv2.imread(str(mp),0)
    facts={'observed_md5':md5(op),'mask_md5':md5(mp),'clean_md5':md5(cp)}
    for dilation in DILATIONS:
        item_id=f'{page}|d{dilation}'; fp=object_hash({'page':page,'observed':facts['observed_md5'],'mask':facts['mask_md5'],'dilation':dilation,'settings':settings_hash})
        cache=RECON_CACHE/f'{page}_d{dilation}_{fp}.npz'; mask=prepare_mask(seed,dilation)
        hit=not FORCE_REBUILD_RECONSTRUCTION and cache.exists() and valid_npz(cache,['mask',*METHODS])
        try:
            started=time.perf_counter()
            if not hit:
                outputs=reconstruct_production(observed,mask)
                for recon in outputs.values(): assert np.array_equal(recon[mask==0],observed[mask==0])
                save_npz_atomic(cache,mask=mask,**outputs)
            elapsed=time.perf_counter()-started
            old=state['items'].get(item_id,{})
            state['items'][item_id]={'page':page,'dilation':dilation,'fingerprint':fp,'source_facts':facts,'reconstruction_complete':True,
                'reconstruction_cache':str(cache),'evaluation_complete':bool(old.get('evaluation_complete',False)),
                'evaluation_cache':old.get('evaluation_cache'),'updated_utc':datetime.now(timezone.utc).isoformat()}
            write_json_atomic(STATE_PATH,state)
            production_rows.append({'page':page,'dilation':dilation,'fingerprint':fp,'cache':'HIT' if hit else 'MISS','seconds':elapsed,
                'mask_pixels':int(np.sum(mask>0)),'reconstruction_cache':str(cache)})
            print(f'Phase A {pi}/{len(valid_pages)} {page} d{dilation}:', 'HIT' if hit else 'MISS')
        except Exception as e:
            state['items'][item_id]={'page':page,'dilation':dilation,'fingerprint':fp,'reconstruction_complete':False,'error':str(e)}
            state['failures'].append({'phase':'A','page':page,'dilation':dilation,'error':str(e),'utc':datetime.now(timezone.utc).isoformat()}); write_json_atomic(STATE_PATH,state)
            print('FAILED A',page,dilation,e)
pd.DataFrame(production_rows).to_csv(TABLES/'production_manifest.csv',index=False)


## Evaluation functions


In [ ]:
def printed_ink(image): return cv2.adaptiveThreshold(image,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge_map(image): return cv2.Canny(image,70,170)>0
def tolerant_f1(ref,pred,region,tol=1):
    r=ref&region; p=pred&region
    if r.sum()==0: return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); rd=cv2.dilate(r.astype(np.uint8),k)>0; pd=cv2.dilate(p.astype(np.uint8),k)>0
    pr=float(np.sum(p&rd)/max(p.sum(),1)); rc=float(np.sum(r&pd)/r.sum()); return pr,rc,float(2*pr*rc/max(pr+rc,1e-12))
def classify_gt(clean,mask):
    ink=printed_ink(clean); long=cv2.morphologyEx(ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0; near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0; m=mask>0
    return {'all':m,'background':m&~ink,'staff':m&staff,'symbol':m&ink&~near,'mixed':m&ink&near}
def score(clean,recon,region):
    if not np.any(region): return None
    d=clean[region].astype(np.float32)-recon[region].astype(np.float32); mse=float(np.mean(d*d)); _,sm=ssim(clean,recon,data_range=255,full=True)
    ip,ir,if1=tolerant_f1(printed_ink(clean),printed_ink(recon),region); ep,er,ef1=tolerant_f1(edge_map(clean),edge_map(recon),region)
    smudge=float(np.sum((recon<180)&(clean>=220)&region)/max(np.sum((clean>=220)&region),1))
    return {'pixels':int(region.sum()),'mae':float(np.mean(np.abs(d))),'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),
            'ssim':float(sm[region].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1_tolerant':ef1,'dark_smudge_rate':smudge}


## Phase B: atomic evaluation checkpoints


In [ ]:
metric_rows=[]; component_rows=[]
for pi,page in enumerate(valid_pages,1):
    cp=find_local(LCLEAN,page); op=find_local(LOBSERVED,page); clean=cv2.imread(str(cp),0); observed=cv2.imread(str(op),0)
    for dilation in DILATIONS:
        item_id=f'{page}|d{dilation}'; item=state['items'].get(item_id,{})
        if not item.get('reconstruction_complete'): print('BLOCKED B',item_id); continue
        efp=object_hash({'reconstruction':item['fingerprint'],'clean_md5':md5(cp),'evaluation_version':'7.1','min_component':MIN_METRIC_COMPONENT_AREA})
        cache=METRIC_CACHE/f'{page}_d{dilation}_{efp}.json'; hit=not FORCE_REBUILD_EVALUATION and cache.exists(); payload=read_json(cache,None) if hit else None
        if not payload or payload.get('evaluation_fingerprint')!=efp: hit=False
        try:
            if not hit:
                rcache=Path(item['reconstruction_cache'])
                if not valid_npz(rcache,['mask',*METHODS]): raise RuntimeError('Invalid reconstruction cache '+str(rcache))
                with np.load(rcache) as z: mask=z['mask']; outputs={m:z[m] for m in METHODS}
                regions=classify_gt(clean,mask); local_m=[]; local_c=[]; ink=printed_ink(clean)
                for method,recon in outputs.items():
                    assert np.array_equal(recon[mask==0],observed[mask==0])
                    for rn,region in regions.items():
                        q=score(clean,recon,region)
                        if q: local_m.append({'page':page,'dilation':dilation,'method':method,'region':rn,**q})
                n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
                for component in range(1,n):
                    area=int(stats[component,cv2.CC_STAT_AREA])
                    if area<MIN_METRIC_COMPONENT_AREA: continue
                    comp=labels==component; ink_fraction=float(np.mean(ink[comp])); category='background' if ink_fraction<.05 else ('structure' if ink_fraction>.30 else 'mixed')
                    x,y,w,h=map(int,stats[component,:4])
                    for method,recon in outputs.items():
                        q=score(clean,recon,comp)
                        if q: local_c.append({'page':page,'dilation':dilation,'component':component,'category':category,'component_area':area,
                            'x':x,'y':y,'w':w,'h':h,'aspect_ratio':max(w,h)/max(min(w,h),1),'clean_ink_fraction':ink_fraction,'method':method,**q})
                payload={'version':'7.1','page':page,'dilation':dilation,'evaluation_fingerprint':efp,'metric_rows':local_m,'component_rows':local_c}
                write_json_atomic(cache,payload)
            metric_rows.extend(payload['metric_rows']); component_rows.extend(payload['component_rows'])
            state['items'][item_id].update({'evaluation_complete':True,'evaluation_fingerprint':efp,'evaluation_cache':str(cache),'updated_utc':datetime.now(timezone.utc).isoformat()}); write_json_atomic(STATE_PATH,state)
            print(f'Phase B {pi}/{len(valid_pages)} {page} d{dilation}:','HIT' if hit else 'MISS')
        except Exception as e:
            state['items'][item_id].update({'evaluation_complete':False,'evaluation_error':str(e)}); state['failures'].append({'phase':'B','page':page,'dilation':dilation,'error':str(e),'utc':datetime.now(timezone.utc).isoformat()}); write_json_atomic(STATE_PATH,state)
            print('FAILED B',page,dilation,e)
results=pd.DataFrame(metric_rows); components=pd.DataFrame(component_rows)
if results.empty: raise RuntimeError('No evaluation records available.')
results.to_csv(TABLES/'metrics_by_page_region.csv',index=False); components.to_csv(TABLES/'component_metrics.csv',index=False)


## Summaries and paired bootstrap comparisons


In [ ]:
summary=(results.groupby(['dilation','method','region'],as_index=False).agg(pages=('page','nunique'),pixels=('pixels','sum'),mae=('mae','mean'),psnr=('psnr','mean'),ssim=('ssim','mean'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1_tolerant=('edge_f1_tolerant','mean'),dark_smudge_rate=('dark_smudge_rate','mean')))
summary.to_csv(TABLES/'summary_by_region.csv',index=False)
def bootstrap(v,seed=42):
    v=np.asarray(v,float); v=v[np.isfinite(v)]
    if not len(v): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed); samples=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(BOOTSTRAP_SAMPLES)])
    return float(v.mean()),float(np.percentile(samples,2.5)),float(np.percentile(samples,97.5))
paired=[]
for dilation in DILATIONS:
    for region,metric,higher in [('all','mae',False),('all','ssim',True),('background','mae',False),('background','dark_smudge_rate',False),('staff','ink_recall',True),('symbol','edge_f1_tolerant',True),('mixed','edge_f1_tolerant',True)]:
        p=results[(results.dilation==dilation)&(results.region==region)].pivot(index='page',columns='method',values=metric)
        if BASELINE not in p: continue
        for method in METHODS:
            if method==BASELINE or method not in p: continue
            d=(p[method]-p[BASELINE])*(1 if higher else -1); mean,lo,hi=bootstrap(d.dropna())
            paired.append({'dilation':dilation,'region':region,'metric':metric,'method':method,'baseline':BASELINE,'benefit_mean':mean,'ci95_low':lo,'ci95_high':hi,'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum())})
paired_df=pd.DataFrame(paired); paired_df.to_csv(TABLES/'paired_method_deltas.csv',index=False)
display(summary); display(paired_df)


## Contact sheets and worst background-smudge crops


In [ ]:
def load_item(page,dilation):
    clean=cv2.imread(str(find_local(LCLEAN,page)),0); observed=cv2.imread(str(find_local(LOBSERVED,page)),0); item=state['items'][f'{page}|d{dilation}']
    with np.load(item['reconstruction_cache']) as z: mask=z['mask']; outputs={m:z[m] for m in METHODS}
    return clean,observed,mask,outputs
sizes=pd.read_csv(TABLES/'production_manifest.csv').groupby('page',as_index=False).mask_pixels.max().sort_values('mask_pixels',ascending=False)
for page in sizes.head(MAX_CONTACT_PAGES).page:
    dilation=2 if 2 in DILATIONS else DILATIONS[0]; clean,observed,mask,outputs=load_item(page,dilation)
    overlay=cv2.cvtColor(observed,cv2.COLOR_GRAY2BGR); layer=overlay.copy(); layer[mask>0]=(0,0,255); overlay=cv2.addWeighted(overlay,.68,layer,.32,0)
    images=[clean,observed,overlay]+[outputs[m] for m in METHODS]; names=['clean','observed','mask']+METHODS
    fig,axes=plt.subplots(2,4,figsize=(24,14))
    for ax,image,name in zip(axes.ravel(),images,names): ax.imshow(image if image.ndim==2 else cv2.cvtColor(image,cv2.COLOR_BGR2RGB),cmap='gray' if image.ndim==2 else None,vmin=0 if image.ndim==2 else None,vmax=255 if image.ndim==2 else None); ax.set_title(name); ax.axis('off')
    for ax in axes.ravel()[len(images):]: ax.axis('off')
    plt.tight_layout(); plt.savefig(CONTACTS/f'{page}_d{dilation}.png',dpi=150,bbox_inches='tight'); plt.close(fig)
base=components[(components.method==BASELINE)&(components.category=='background')].sort_values(['dark_smudge_rate','component_area'],ascending=[False,False]).head(MAX_FAILURE_CROPS)
for rank,(_,r) in enumerate(base.iterrows(),1):
    page=str(r.page); dilation=int(r.dilation); clean,observed,mask,outputs=load_item(page,dilation); H,W=clean.shape; pad=32
    x0=max(0,int(r.x)-pad); y0=max(0,int(r.y)-pad); x1=min(W,int(r.x+r.w)+pad); y1=min(H,int(r.y+r.h)+pad)
    panels=[clean[y0:y1,x0:x1],observed[y0:y1,x0:x1]]+[outputs[m][y0:y1,x0:x1] for m in METHODS]; names=['clean','observed']+METHODS
    fig,axes=plt.subplots(1,len(panels),figsize=(4*len(panels),4))
    for ax,image,name in zip(axes,panels,names): ax.imshow(image,cmap='gray',vmin=0,vmax=255); ax.set_title(name); ax.axis('off')
    plt.tight_layout(); plt.savefig(FAILURES/f'{rank:03d}_{page}_d{dilation}_c{int(r.component)}.png',dpi=170,bbox_inches='tight'); plt.close(fig)


## Decision table, manifest and resume audit


In [ ]:
decision=[]
for dilation in DILATIONS:
    for method in METHODS:
        if method==BASELINE: continue
        def row(region,metric): return paired_df[(paired_df.dilation==dilation)&(paired_df.region==region)&(paired_df.metric==metric)&(paired_df.method==method)]
        mae=row('all','mae'); smudge=row('background','dark_smudge_rate'); mixed=row('mixed','edge_f1_tolerant')
        decision.append({'dilation':dilation,'method':method,'all_mae_benefit':float(mae.iloc[0].benefit_mean) if len(mae) else np.nan,
            'all_mae_ci_low':float(mae.iloc[0].ci95_low) if len(mae) else np.nan,'background_smudge_benefit':float(smudge.iloc[0].benefit_mean) if len(smudge) else np.nan,
            'background_smudge_ci_low':float(smudge.iloc[0].ci95_low) if len(smudge) else np.nan,'mixed_edge_benefit':float(mixed.iloc[0].benefit_mean) if len(mixed) else np.nan,
            'candidate_for_production':bool(len(mae) and len(smudge) and mae.iloc[0].benefit_mean>0 and smudge.iloc[0].benefit_mean>0)})
decision_df=pd.DataFrame(decision).sort_values(['candidate_for_production','background_smudge_benefit','all_mae_benefit'],ascending=[False,False,False]); decision_df.to_csv(TABLES/'production_decision_table.csv',index=False)
status=pd.DataFrame([{'item_id':k,**v} for k,v in state['items'].items()]); status.to_csv(TABLES/'resume_status.csv',index=False)
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.1','pages_discovered':len(discovered),'pages_evaluated':results.page.nunique(),'methods':METHODS,'dilations':DILATIONS,
          'settings':settings,'guarantees':{'clean_not_passed_to_reconstruction':True,'outside_mask_immutable':True,'atomic_reconstruction_checkpoint':True,'atomic_evaluation_checkpoint':True,'duplicate_base_inpainting_removed':True},
          'resume':{'state':str(STATE_PATH),'reconstruction_cache':str(RECON_CACHE),'evaluation_cache':str(METRIC_CACHE),'tracked_items':len(state['items']),'reconstruction_complete':int(status.reconstruction_complete.fillna(False).sum()),'evaluation_complete':int(status.evaluation_complete.fillna(False).sum())}}
(OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2))
state['last_completed_run']=str(OUT); state['updated_utc']=datetime.now(timezone.utc).isoformat(); write_json_atomic(STATE_PATH,state)
display(decision_df); print('V7.1 COMPLETE:',OUT); print('Persistent state:',STATE_PATH)
